# Advanced Feature Engineering

This notebook focuses on transforming raw Titanic passenger data into a robust machine learning-ready dataset.

The feature engineering process aims to:
- extract hidden demographic patterns
- encode socioeconomic information
- model passenger group behavior
- improve nonlinear feature interactions
- handle missing values intelligently
- prepare data for ensemble learning models

Feature engineering is one of the most critical components of high-performing tabular machine learning systems.

In [31]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import sklearn 

In [32]:
train_df=pd.read_csv("/kaggle/input/competitions/titanic/train.csv")
test_df=pd.read_csv("/kaggle/input/competitions/titanic/test.csv")

In [33]:
train_df['is_train'] = 1
test_df['is_train'] = 0
test_df['Survived'] = np.nan

full_df = pd.concat([train_df, test_df], axis=0)

train_df = full_df[full_df['is_train'] == 1].copy()
test_df = full_df[full_df['is_train'] == 0].copy()

train_df.drop(columns=['is_train'], inplace=True)
test_df.drop(columns=['is_train', 'Survived'], inplace=True)

# concatination 
This prevents:

- inconsistent encodings
- category mismatch

In [34]:
full_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,is_train
0,1,0.0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,1
1,2,1.0,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,1
2,3,1.0,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,1
3,4,1.0,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S,1
4,5,0.0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S,1


# Title feature

In [35]:
full_df['Title'] = full_df['Name'].str.extract(
    ' ([A-Za-z]+)\.',
    expand=False
)

# Missing values handeling

## Age Imputation using:

- Title
- Pclass
- Sex

In [36]:
full_df['Age'] = full_df.groupby(
    ['Title', 'Pclass', 'Sex']
)['Age'].transform(
    lambda x: x.fillna(x.median())
)

## Final fallback age imputation

In [37]:
full_df['Age'] = full_df['Age'].fillna(
    full_df['Age'].median()
)

## Missing embarkation values are replaced using the most frequent embarkation port (mode)

In [38]:
full_df['Embarked'] = full_df['Embarked'].fillna(
    full_df['Embarked'].mode()[0]
)

## Fare imputation by (median)

In [39]:
full_df['Fare'] = full_df.groupby(
    'Pclass'
)['Fare'].transform(
    lambda x: x.fillna(x.median())
)

# Family features

In [40]:
full_df['FamilySize'] = (
    full_df['SibSp'] +
    full_df['Parch'] + 1
)

full_df['IsAlone'] = (
    full_df['FamilySize'] == 1
).astype(int)

# Ticket Group size
Passengers sharing the same ticket likely traveled together.


In [41]:
full_df['TicketGroup'] = full_df.groupby(
    'Ticket'
)['Ticket'].transform('count')

# Cabin Deck Extraction

In [42]:
full_df['Deck'] = full_df['Cabin'].str[0]

full_df['Deck'] = full_df['Deck'].fillna('U')

# Fare Per Person

In [43]:
full_df['FarePerPerson'] = (
    full_df['Fare'] /
    full_df['FamilySize']
)

# Age groups

In [44]:
full_df['AgeBin'] = pd.cut(
    full_df['Age'],
    bins=[0,12,18,35,60,100],
    labels=[
        'Child',
        'Teen',
        'Young Adult',
        'Adult',
        'Senior'
    ]
)

# Interaction Features

In [45]:
# Sex + Passenger Class
full_df['Sex_Pclass'] = (
    full_df['Sex'] + "_" +
    full_df['Pclass'].astype(str)
)

In [46]:
# Age * Class
full_df['Age_Class'] = (
    full_df['Age'] *
    full_df['Pclass']
)

In [47]:
# Fare * Class
full_df['Fare_Class'] = (
    full_df['Fare'] *
    full_df['Pclass']
)

In [48]:
# Family * Fare
full_df['Family_Fare'] = (
    full_df['FamilySize'] *
    full_df['Fare']
)

In [49]:
full_df.isnull().sum()

PassengerId         0
Survived          418
Pclass              0
Name                0
Sex                 0
Age                 0
SibSp               0
Parch               0
Ticket              0
Fare                0
Cabin            1014
Embarked            0
is_train            0
Title               0
FamilySize          0
IsAlone             0
TicketGroup         0
Deck                0
FarePerPerson       0
AgeBin              0
Sex_Pclass          0
Age_Class           0
Fare_Class          0
Family_Fare         0
dtype: int64

In [50]:
processed_train = full_df[full_df['is_train'] == 1].copy()
processed_test = full_df[full_df['is_train'] == 0].copy()

In [51]:
processed_train.drop(columns=['is_train']).to_csv(
    "processed_train.csv",
    index=False
)

processed_test.drop(columns=['is_train', 'Survived'], errors='ignore').to_csv(
    "processed_test.csv",
    index=False
)